In [ ]:
# Cell 1: W&B + mamba-ssm install
import os, sys
os.environ.setdefault("WANDB_MODE", "online")
os.environ.setdefault("WANDB_SILENT", "true")
import wandb
api_key = os.environ.get("WANDB_API_KEY")
if api_key:
    wandb.login(key=api_key, relogin=True)
    print("W&B ready")
else:
    os.environ["WANDB_MODE"] = "offline"
    print("WANDB_API_KEY not set; running in offline mode")

print("Installing causal-conv1d...")
os.system("pip install causal-conv1d --no-build-isolation -q 2>&1 | tail -2")

print("Building mamba-ssm (first build can take ~15 min)...")
os.system("""
rm -rf /tmp/mamba_build
cp -r /kaggle/input/datasets/mbrosseau/mamba-ssm/mamba-main/mamba-main /tmp/mamba_build
pip install /tmp/mamba_build --no-build-isolation -q 2>&1 | tail -3
""")

os.system("pip install transformers==4.44.2 -q 2>&1 | tail -2")

# Reload and verify
for mod in list(sys.modules.keys()):
    if "mamba" in mod or "transformers" in mod:
        del sys.modules[mod]

try:
    import mamba_ssm
    from mamba_ssm import selective_scan_fn
    print(f"mamba-ssm {mamba_ssm.__version__} ready")
except Exception as e:
    print(f"mamba-ssm unavailable: {e} (PyTorch fallback will be used)")

In [ ]:
# Cell 2: Copy files + HDF5
import os, sys, shutil

DATASET = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "transforms.py" in files:
        DATASET = root
        break
assert DATASET, "Add your code dataset as input"
print(f"Dataset: {DATASET}")

FILES = [
    "transforms.py", "rothermel.py", "trainer.py", "wildfire_dataset.py",
    "pgss_block.py", "notebook_1_resnet_unet.py", "notebook_5_pi_vmunet_phase3b.py"
 ]
for f in FILES:
    src = os.path.join(DATASET, f)
    if os.path.exists(src):
        shutil.copy(src, f"/kaggle/working/{f}")
        print(f"Copied: {f}")
    else:
        print(f"Missing: {f}")

os.makedirs("/kaggle/working/outputs/checkpoints", exist_ok=True)
sys.path.insert(0, "/kaggle/working")
print("Copy done")

import h5py, tensorflow as tf, numpy as np
from tqdm.auto import tqdm

HDF5_PATH = "/kaggle/working/wildfire_data.h5"
needs_rebuild = True
if os.path.exists(HDF5_PATH):
    with h5py.File(HDF5_PATH, "r") as f:
        keys = list(f.keys())
    if all(s in keys for s in ["train", "eval", "test"]):
        print(f"HDF5 complete: {os.path.getsize(HDF5_PATH)/1e9:.2f} GB")
        needs_rebuild = False
    else:
        print(f"Incomplete splits {keys}; rebuilding")
        os.remove(HDF5_PATH)

if needs_rebuild:
    TFREC_DIR = None
    for root, dirs, files in os.walk("/kaggle/input"):
        if any(f.endswith(".tfrecord") for f in files):
            TFREC_DIR = root
            break
    assert TFREC_DIR, "TFRecord dataset not found"
    print(f"TFRecords: {TFREC_DIR}")

    CHANNELS = ["elevation","th","vs","tmmn","tmmx","sph","pr",
                "pdsi","NDVI","population","erc","PrevFireMask"]
    SZ = 64
    FEAT = {n: tf.io.FixedLenFeature([SZ, SZ], tf.float32) for n in CHANNELS + ["FireMask"]}

    def parse(s):
        p = tf.io.parse_single_example(s, FEAT)
        return tf.stack([p[c] for c in CHANNELS], axis=-1), p["FireMask"]

    all_files = sorted([os.path.join(TFREC_DIR, f)
                        for f in os.listdir(TFREC_DIR) if "tfrecord" in f])
    splits = {s: [f for f in all_files if f"_{s}_" in os.path.basename(f)]
              for s in ("train", "eval", "test")}
    for s, fs in splits.items():
        print(f"{s}: {len(fs)} shards")

    with h5py.File(HDF5_PATH, "w") as hf:
        for split, sfiles in splits.items():
            g = hf.require_group(split)
            di = g.create_dataset("inputs", shape=(0, SZ, SZ, 12), maxshape=(None, SZ, SZ, 12),
                                  dtype="float32", chunks=(32, SZ, SZ, 12),
                                  compression="gzip", compression_opts=4)
            dt = g.create_dataset("targets", shape=(0, SZ, SZ), maxshape=(None, SZ, SZ),
                                  dtype="float32", chunks=(32, SZ, SZ),
                                  compression="gzip", compression_opts=4)
            ds = (tf.data.TFRecordDataset(sfiles)
                  .map(parse, num_parallel_calls=tf.data.AUTOTUNE)
                  .batch(128).prefetch(2))
            idx = 0
            for ib, tb in tqdm(ds, desc=f"{split}"):
                n = ib.shape[0]
                di.resize(idx + n, axis=0); dt.resize(idx + n, axis=0)
                di[idx:idx+n] = ib.numpy(); dt[idx:idx+n] = tb.numpy()
                idx += n
            print(f"{split}: {idx} samples")
    print(f"HDF5 saved: {os.path.getsize(HDF5_PATH)/1e9:.2f} GB")

In [ ]:
# Cell 3: ResNet-UNet (clean rerun)
for mod in list(sys.modules.keys()):
    if any(x in mod for x in ["wildfire_dataset", "trainer", "transforms", "rothermel", "pgss"]):
        del sys.modules[mod]
exec(open("/kaggle/working/notebook_1_resnet_unet.py").read())

In [ ]:
# Cell 4: PI-VMUNet Phase 4 (PDE loss)
import gc, torch
try:
    del model, trainer, optimizer, train_dl, val_dl, test_dl, scheduler, loss_fn, config
except Exception:
    pass
gc.collect()
torch.cuda.empty_cache()

for mod in list(sys.modules.keys()):
    if any(x in mod for x in ["wildfire_dataset", "trainer", "transforms", "rothermel", "pgss"]):
        del sys.modules[mod]
exec(open("/kaggle/working/notebook_5_pi_vmunet_phase3b.py").read())